# 07 Automated testing

- Automated testing - what and why
- Unittest
- Testing UI Application
- Testing Strategies & Resources

Ref:
- [Python unittest](https://docs.python.org/3/library/unittest.html) documentation
- PySide6 Testing
- Tkinter Testing
- Real Python's [Python Testing Tutorial](https://realpython.com/python-testing/)
- [PyTest](https://docs.pytest.org/en/stable/) Documentation (alternative testing framework)

# Automated testing - what and why

**What?**
- Code that verifies other code works correctly
- Runs automatically without manual intervention
>- No lots of clicking and typing tens of inputs
- Confirms software behaves as expected
- Acts as a safety net when making changes

It might seems like an extra work at first, but this will save time in a long run.

**Why?**
- Catch bugs early
>- Example: Adding a feature that accidentally breaks something else

- Document expected behavior
>- Tests show how code should work
>-New team members can understand code by reading tests

- Safe refactoring
>- Change implementation while preserving behavior
>- Example: Optimizing a sorting algorithm

- Reduce debugging time
>- Tests pinpoint exactly where things broke
>- No more "it worked yesterday" problems

**When?**

Short ans: you should run tests regularly, esp when changing program logic and stucture.

In another words, **run tests whenever you can** ^^;





**Examples of automated testing usage**

1) Multiple developers working on the same project
- Your code depends on a function someone else wrote
- They change their function, breaking your code
- Tests alert everyone before the bug reaches users

2) Cost comparison
- Bug fixed during development: ~1 hour
- Bug fixed after release: ~10-100 hours + reputation damage
- Example: A banking app calculation error

# Python Unittest



## Unittest structure

- Usually, we write tests in a separate file called `test_*.py`

- unittest common stucture

```
import unittest
from my_module import my_function

class TestCalculator(unittest.TestCase):
    
    def setUp(self):
        # This runs before each test
        pass
        
    def tearDown(self):
        # This runs after each test
        pass
    
    def test_addition(self):
        # Actual test code
        pass
```

### `setUp()`
- run before each test
- Instead of repeating initialization code in every test method, you can centralize it in setUp()
- Each test starts with the same baseline conditions, making tests more reliable.
- Each test method should be independent of others. These methods help ensure test cases don't affect each other.


For example,
```
self.account = BankAccount(initial_balance=1000)
```

###`tearDown()`
- runs after each test
- Ensures resources are properly released after each test, preventing resource leaks.

For example,
  ```
  self.account.close()
  ```



Without `setUp()` and `tearDown()`, you'd need to repeat the account creation and closing in every test method, making your code more verbose and error-prone.

###`test_something()`

- the actual test function
- there should be several of these functions


For example,

    ```
    def test_deposit(self):
      self.account.deposit(500)
      self.assertEqual(self.account.balance, 1500)
    ```
    
    ```
    def test_withdrawal(self):
      self.account.withdraw(300)
      self.assertEqual(self.account.balance, 700)
    ```

## Test results

The program will display a symbol for each test function.

- `.` = Test passed
- `F` = Test failed (throwing an assertion error)
- `E` = Test error (exception raised)
- `s` = Test skipped

**Pass**
```
.........
----------------------------------------------------------------------
Ran 9 tests in 0.001s

OK
```

**Failed**
```
F...FF.F.
======================================================================
FAIL: test_account_close_resets_balance (__main__.BankAccountTest.test_account_close_resets_balance)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "g:\My Drive\Classes - current\IxProg Interaction Programming\Lecture slides\test_bank_account.py", line 56, in test_account_close_resets_balance
    self.assertEqual(self.account.balance, 0)
AssertionError: 1000 != 0

======================================================================
FAIL: test_deposit_negative_amount (__main__.BankAccountTest.test_deposit_negative_amount)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "g:\My Drive\Classes - current\IxProg Interaction Programming\Lecture slides\test_bank_account.py", line 27, in test_deposit_negative_amount
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised

###################
# List of other FAILs
# I removed other FAILs here due to space limitation
###################

----------------------------------------------------------------------
Ran 9 tests in 0.004s

FAILED (failures=4)
```

Try running these codes and fix the code where appropriate

In [ ]:
# bank_account.py
class BankAccount:
    def __init__(self, initial_balance=0):
        # Bug 1: Doesn't validate if initial_balance is negative
        self.balance = initial_balance
        self.is_open = True

    def deposit(self, amount):
        if not self.is_open:
            raise ValueError("Cannot deposit to a closed account")
        # Bug 2: Missing validation for positive amount
        # Should have: if amount <= 0: raise ValueError(...)
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if not self.is_open:
            raise ValueError("Cannot withdraw from a closed account")
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        # Bug 3: No check for sufficient funds
        # Should have: if amount > self.balance: raise ValueError(...)
        self.balance -= amount
        return self.balance

    def close(self):
        # Bug 4: Doesn't reset balance to zero when closing account
        self.is_open = False

In [ ]:
# test_bank_account.py
import unittest
from bank_account import BankAccount

class BankAccountTest(unittest.TestCase):
    def setUp(self):
        # Run before each test
        self.account = BankAccount(initial_balance=1000)

    def tearDown(self):
        # Run after each test
        self.account.close()

    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 1000)

        # Test for Bug 1: Should not allow negative initial balance
        with self.assertRaises(ValueError):
            negative_account = BankAccount(initial_balance=-100)

    def test_deposit(self):
        self.account.deposit(500)
        self.assertEqual(self.account.balance, 1500)

    def test_deposit_negative_amount(self):
        # Test for Bug 2: Should not allow negative deposits
        with self.assertRaises(ValueError):
            self.account.deposit(-100)

    def test_withdrawal(self):
        self.account.withdraw(300)
        self.assertEqual(self.account.balance, 700)

    def test_withdrawal_negative_amount(self):
        with self.assertRaises(ValueError):
            self.account.withdraw(-50)

    def test_withdrawal_insufficient_funds(self):
        # Test for Bug 3: Should not allow withdrawals exceeding balance
        with self.assertRaises(ValueError):
            self.account.withdraw(2000)

    def test_closed_account_deposit(self):
        self.account.close()
        with self.assertRaises(ValueError):
            self.account.deposit(500)

    def test_closed_account_withdrawal(self):
        self.account.close()
        with self.assertRaises(ValueError):
            self.account.withdraw(200)

    def test_account_close_resets_balance(self):
        # Test for Bug 4: Balance should be zero after closing
        self.account.close()
        self.assertEqual(self.account.balance, 0)


if __name__ == "__main__":
    unittest.main()


Note:
The `unittest.main()` function automatically discovers and runs all test cases defined in your module through a process called test discovery.

Here's how it works:

- Class Detection:
>- When you call `unittest.main()`, it looks for all classes in the current module that inherit from `unittest.TestCase`
- Method Detection:
>- Within each TestCase class it finds, it automatically identifies all methods that start with the prefix `test_` as test methods that should be run.
- Test Suite Creation:
>- It automatically builds a test suite consisting of all these discovered test methods.
- Test Runner:
>- It then creates a test runner that executes all the tests in the suite.

## Assertions

- Methods of the `unittest.TestCase` class
- Provide more informative error messages, making it easier to diagnose test failures.

**Common assertions**

- `assertEqual(a, b)` - Verify that a equals b
- `assertTrue(x)` - Verify that x is True
- `assertFalse(x)` - Verify that x is False
- `assertRaises(exception, callable, *args)` - Verify that calling a function raises an exception
- `assertIn(a, b)` - Verify that a is in b (list, tuple, etc.)
- `assertIsInstance(a, Type)` - Verify that a is an instance of Type

(More in the examples and references)

Ref:
- https://docs.python.org/3/library/unittest.html#assert-methods
- https://docs.python.org/3/library/unittest.html#unittest.TestCase

###Testing Values

In [ ]:
import unittest

class TestAssertionExamples(unittest.TestCase):

    def test_assertEqual(self):
        # assertEqual(a, b) - Verify that a equals b
        result = 5 + 5
        expected = 10
        self.assertEqual(result, expected, "Addition should equal 10")

    def test_assertTrue(self):
        # assertTrue(x) - Verify that x is True
        is_valid = len("hello") > 0
        self.assertTrue(is_valid, "String should have length greater than 0")

    def test_assertFalse(self):
        # assertFalse(x) - Verify that x is False
        is_empty = len([]) > 0
        self.assertFalse(is_empty, "Empty list length should not be greater than 0")

    def test_assertRaises(self):
        # assertRaises(exception, callable, *args) - Verify that calling a function raises an exception
        def divide_by_zero():
            return 1/0
        self.assertRaises(ZeroDivisionError, divide_by_zero)

    def test_assertIn(self):
        # assertIn(a, b) - Verify that a is in b (list, tuple, etc.)
        fruits = ["apple", "banana", "cherry"]
        self.assertIn("banana", fruits, "banana should be in the fruits list")

    def test_assertIsInstance(self):
        # assertIsInstance(a, Type) - Verify that a is an instance of Type
        value = "test string"
        self.assertIsInstance(value, str, "value should be a string")

    def test_assertNotEqual(self):
        # assertNotEqual(a, b) - Verify that a does not equal b
        self.assertNotEqual(5, 10, "5 should not equal 10")

    def test_assertIs(self):
        # assertIs(a, b) - Verify that a is b (same object identity)
        a = [1, 2, 3]
        b = a  # b references the same object as a
        self.assertIs(a, b, "a and b should reference the same object")

    def test_assertIsNot(self):
        # assertIsNot(a, b) - Verify that a is not b (different object identity)
        a = [1, 2, 3]
        b = [1, 2, 3]  # Same content but different object
        self.assertIsNot(a, b, "a and b should be different objects")

    def test_assertNotIn(self):
        # assertNotIn(a, b) - Verify that a is not in b
        numbers = [1, 2, 3, 4, 5]
        self.assertNotIn(10, numbers, "10 should not be in the numbers list")

    def test_assertIsNone(self):
        # assertIsNone(x) - Verify that x is None
        value = None
        self.assertIsNone(value, "value should be None")

    def test_assertIsNotNone(self):
        # assertIsNotNone(x) - Verify that x is not None
        value = "something"
        self.assertIsNotNone(value, "value should not be None")

    def test_assertAlmostEqual(self):
        # assertAlmostEqual(a, b) - Verify that a and b are almost equal (for floating point)
        self.assertAlmostEqual(0.1 + 0.2, 0.3, places=7, msg="Floating point addition should be almost equal")

    def test_assertGreater(self):
        # assertGreater(a, b) - Verify that a > b
        self.assertGreater(10, 5, "10 should be greater than 5")

    def test_assertGreaterEqual(self):
        # assertGreaterEqual(a, b) - Verify that a >= b
        self.assertGreaterEqual(10, 10, "10 should be greater than or equal to 10")

    def test_assertLess(self):
        # assertLess(a, b) - Verify that a < b
        self.assertLess(5, 10, "5 should be less than 10")

    def test_assertLessEqual(self):
        # assertLessEqual(a, b) - Verify that a <= b
        self.assertLessEqual(5, 5, "5 should be less than or equal to 5")

    def test_assertRegex(self):
        # assertRegex(text, regex) - Verify that regex matches text
        self.assertRegex("hello world", r"hello", "Text should match the regex pattern")

    def test_assertCountEqual(self):
        # assertCountEqual(a, b) - Verify that a and b have the same elements (ignoring order)
        self.assertCountEqual([1, 2, 3], [3, 1, 2], "Lists should have the same elements regardless of order")

if __name__ == '__main__':
    unittest.main()

### Testing Exceptions

Test that your code raises the expected exceptions when it should be

```
with self.assertRaises(ExpectedException):
    # Code that should raise ExpectedException
```

E.g.
```
with self.assertRaises(TypeError):
    calculate_area("4", 5)
```
- This test verifies that calling the `calculate_area()` function with a string "4" and an integer 5 will raise a `TypeError`.
- If the expected exception (in this case `TypeError`) is raised during execution of the block, the test passes
- If no exception is raised, or if a different exception type is raised, the test fails

In [ ]:
import unittest

def calculate_area(length, width):
    if not isinstance(length, (int, float)) or not isinstance(width, (int, float)):
        raise TypeError("Both length and width must be numbers")
    return length * width

class TestCalculateArea(unittest.TestCase):
    def test_invalid_input_types(self):
        # Test that string inputs raise TypeError
        with self.assertRaises(TypeError):
            calculate_area("4", 5)

        # You can have multiple assertions in one test
        with self.assertRaises(TypeError):
            calculate_area(4, "5")

        # You can also test the error message
        with self.assertRaises(TypeError) as context:
            calculate_area([1, 2], 5)
        self.assertTrue("must be numbers" in str(context.exception))

    def test_valid_inputs(self):
        # Make sure it works with valid inputs
        self.assertEqual(calculate_area(4, 5), 20)
        self.assertEqual(calculate_area(4.5, 5.5), 24.75)

if __name__ == '__main__':
    unittest.main()

## Examples

**Area calculation**

In [ ]:
# area.py
def calculate_area(length, width):
    return length * width

In [ ]:
# test_area.py
import unittest
from area import calculate_area

class TestAreaCalculation(unittest.TestCase):
    def test_positive_numbers(self):
        # Tests with positive numbers (put in several tests here for different cases)
        self.assertEqual(calculate_area(4, 5), 20)

    def test_zero(self):
        # Test with zero (put in several tests here for different cases)
        pass

    def test_negative_numbers(self):
        # Test with negative numbers (put in several tests here for different cases)
        pass

if __name__ == '__main__':
    unittest.main()

**Temperature conversion**

- Create a function celsius_to_fahrenheit(celsius) that converts Celsius to Fahrenheit
>- Put this into `temperature.py`
- Write tests to verify it works correctly
>- Put this into `test_temperature.py`
- Formula: F = C * 9/5 + 32

In [ ]:
# temperature.py
def celsius_to_fahrenheit(celsius):
    # your code here
    return 0 # temperature in F

In [ ]:
# test_temperature.py
import unittest
from temperature import celsius_to_fahrenheit

class TestTemperatureConversion(unittest.TestCase):
    def test_freezing_point(self):
        pass

    def test_boiling_point(self):
        pass

    def test_body_temperature(self):
        pass

    def test_negative_temperature(self):
        pass

    def test_invalid_input(self):
        with self.assertRaises(TypeError):
            celsius_to_fahrenheit("zero")

if __name__ == '__main__':
    unittest.main()

# Testing GUI Application

**Why can't we use unittest? for GUI**

- Event-driven nature
>- Applications respond to user events (clicks, key presses)
>- Events can happen in unpredictable order

- Asynchronous behavior
>- Actions may not have immediate effects
>- Animation and timing issues

- State management complexity
>- UI has multiple states based on user interaction
>- Example: Button enabled/disabled based on input validity




## Model - View - Controller (MVC) Pattern

**Model**: Data and business logic (highly testable)

**View**: Visual elements (harder to test)

**Controller**: Connects model and view (medium testability)

- Make UI testing easier
- Separate your code into testable components

Comparing these 2 codes

In [ ]:
# Bad (hard to test):
class CalculatorApp:
    def __init__(self):
        self.result = 0
        # UI setup code...

    def add_button_clicked(self):
        input_value = int(self.input_field.text())
        self.result += input_value # the logic part
        self.result_label.setText(str(self.result))

In [ ]:
# Better (easier to test):
class Calculator:  # Model
    def __init__(self):
        self.result = 0

    def add(self, value): # Now we can test this "logic" method separately
        self.result += value
        return self.result

class CalculatorApp:  # View + Controller
    def __init__(self):
        self.calculator = Calculator()
        # UI setup code...

    def add_button_clicked(self):
        input_value = int(self.input_field.text())
        new_result = self.calculator.add(input_value) # separate the "logic" part into another method
        self.result_label.setText(str(new_result))

## What and what not to test

**Do**
- Business logic (calculations, data transformations)
- State transitions (what happens when button is clicked)
- Event handlers (do they call the right functions)

**Don't**
- Framework behavior (PySide6 itself)
- Visual appearance (except in specialized UI testing)
- Third-party libraries (Just assume they work correctly)

## QTest

- Testing PySide6 application using QTest
- QTest **simulate events** happening in an application during the test

```
from PySide6.QtTest import QTest
from PySide6.QtCore import Qt
```

Ref:
- https://doc.qt.io/qtforpython-6/PySide6/QtTest/QTest.html
- https://doc.qt.io/qt-6/qtest-tutorial.html
- https://doc.qt.io/qt-6/qtest.html


### Keyboard Events

In [ ]:
# a key press.
QTest.keyPress(widget, key)

# a key release.
QTest.keyRelease(widget, key)

# Send a sequence of key clicks to a widget
QTest.keyClicks(widget, "text to type")

# Key click (combines press and release)
QTest.keyClick(widget, Qt.Key_A, Qt.ControlModifier)

# Keyboard shortcut simulation
QTest.keySequence(widget, QKeySequence("Ctrl+S"))

### Mouse Events

In [ ]:
# a double-click with the specified mouse button.
QTest.mouseDClick(widget, button)

# Mouse move event
QTest.mouseMove(widget, QPoint(x, y))

# Mouse press without release
QTest.mousePress(widget, Qt.LeftButton, Qt.NoModifier, QPoint(x, y))

# Mouse release after press
QTest.mouseRelease(widget, Qt.LeftButton, Qt.NoModifier, QPoint(x, y))

# Mouse click (combines press and release)
QTest.mouseClick(widget, Qt.LeftButton, Qt.NoModifier, QPoint(x, y))

### Delays and Timings

In [ ]:
# Pause the test for a specified number of milliseconds.
QTest.qWait(ms)

# Wait for a signal to be emitted
QTest.qWaitForSignal(object, signal, timeout=5000)

# Wait for a window to show
QTest.qWaitForWindowShown(window)

# Wait until the widget's window becomes active.
QTest.qWaitForWindowActive(widget)

# Wait until the widget's window is exposed.
QTest.qWaitForWindowExposed(widget)

# Wait for a specific condition to be met
QTest.qWait(timeout)

### Event Handling:

In [ ]:
# Send a QEvent to a widget
QTest.sendEvent(widget, event)

# Simulate a context menu event
QTest.mouseEvent(QEvent.MouseButtonRelease, widget, Qt.RightButton, Qt.NoModifier, QPoint(x, y))

## Examples

###A counter application


In [ ]:
# counter.py
from PySide6.QtWidgets import QWidget, QPushButton, QLabel, QVBoxLayout, QApplication
import sys

class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1
        return self.count

    def decrement(self):
        self.count -= 1
        return self.count

class CounterWidget(QWidget):
    def __init__(self):
        super().__init__()
        self.counter = Counter()

        self.label = QLabel(f"Count: {self.counter.count}")
        self.inc_button = QPushButton("Increment")
        self.dec_button = QPushButton("Decrement")

        self.inc_button.clicked.connect(self.increment)
        self.dec_button.clicked.connect(self.decrement)

        layout = QVBoxLayout()
        layout.addWidget(self.label)
        layout.addWidget(self.inc_button)
        layout.addWidget(self.dec_button)
        self.setLayout(layout)

    def increment(self):
        new_count = self.counter.increment()
        self.update_label(new_count)

    def decrement(self):
        new_count = self.counter.decrement()
        self.update_label(new_count)

    def update_label(self, count):
        self.label.setText(f"Count: {count}")

if __name__ == "__main__":
    app = QApplication()
    w = CounterWidget()
    w.show()
    sys.exit(app.exec())


In [ ]:
# test_counter.py
import unittest
from counter import Counter
from PySide6.QtTest import QTest
from PySide6.QtCore import Qt
from counter import CounterWidget
from PySide6.QtWidgets import QApplication
import sys

# Create a QApplication instance for tests
app = QApplication(sys.argv)

class TestCounter(unittest.TestCase):
    def setUp(self):
        self.counter = Counter()

    def test_initial_state(self):
        self.assertEqual(self.counter.count, 0)

    def test_increment(self):
        self.assertEqual(self.counter.increment(), 1)
        self.assertEqual(self.counter.count, 1)

        # Test multiple increments
        self.counter.increment()
        self.assertEqual(self.counter.increment(), 3)
        self.assertEqual(self.counter.count, 3)

    def test_decrement(self):
        self.assertEqual(self.counter.decrement(), -1)
        self.assertEqual(self.counter.count, -1)

        # Test multiple decrements
        self.counter.decrement()
        self.assertEqual(self.counter.decrement(), -3)
        self.assertEqual(self.counter.count, -3)

# In test_counter.py
class TestCounterWidget(unittest.TestCase):
    def setUp(self):
        self.widget = CounterWidget()

    def test_initial_state(self):
        self.assertEqual(self.widget.counter.count, 0)
        self.assertEqual(self.widget.label.text(), "Count: 0")

    def test_increment_button(self):
        # Simulate clicking the increment button
        QTest.mouseClick(self.widget.inc_button, Qt.LeftButton)

        # Check that the counter value and label are updated
        self.assertEqual(self.widget.counter.count, 1)
        self.assertEqual(self.widget.label.text(), "Count: 1")

        # Click two more times
        QTest.mouseClick(self.widget.inc_button, Qt.LeftButton)
        QTest.mouseClick(self.widget.inc_button, Qt.LeftButton)
        self.assertEqual(self.widget.counter.count, 3)
        self.assertEqual(self.widget.label.text(), "Count: 3")

    def test_decrement_button(self):
        # Simulate clicking the decrement button
        QTest.mouseClick(self.widget.dec_button, Qt.LeftButton)

        # Check that the counter value and label are updated
        self.assertEqual(self.widget.counter.count, -1)
        self.assertEqual(self.widget.label.text(), "Count: -1")


if __name__ == '__main__':
    unittest.main()

### Exclusive PySide6 event tests!

[app_event.py](https://drive.google.com/file/d/14cbv-tJFiiXOjZHcj-tvfg745FM_9Csw/view?usp=sharing)

[test_event.py](https://drive.google.com/file/d/14bQPnR4LhN2l7HfGjl_SGZRJ6Fnt5dri/view?usp=sharing)

Download the files and see for yourself!

# Testing Strategies & Resources

## Dev & Test workflow

**When to test**
- Ideally before writing code (Test-Driven Development)
- At minimum, before considering a feature "done"

**How often to run tests**
- Run relevant tests as you develop
- Run all tests before submitting the work
- Automate with CI/CD (beyond scope of this lecture)

**Apply to your projects**
- Start with small, critical functions
- Gradually expand test coverage
- Aim for at least basic test coverage for the work

## Best practice

- Test one thing per test
>- Each test should verify one specific behavior
>- Makes it easier to identify what broke

- Keep tests independent
>- Tests should not depend on each other
>- Should be able to run in any order

- Write readable tests
>- Tests serve as documentation
>- Use descriptive test names: test_empty_list_returns_zero

- Test edge cases
>- Empty inputs
>- Boundary values
>- Invalid inputs

- Don't test the framework
>- Focus on your code, not Qt or tkinter